In [1]:
import pandas as pd
import numpy as np  

In [2]:
df = pd.read_csv('Investment.csv')
df = df.sort_values(by='Date', ascending=True)
print(df.head())

          Symbol        Date    Open    High     Low   Close Percent Change  \
1157  INVESTMENT  2021-03-31  104.80  105.82  104.42  105.82         0.00 %   
1156  INVESTMENT  2021-04-01  107.58  107.58  104.39  104.73         0.00 %   
1155  INVESTMENT  2021-04-04  104.73  104.73  101.78  102.27         0.00 %   
1154  INVESTMENT  2021-04-05  102.51  102.71  100.96  101.41         0.00 %   
1153  INVESTMENT  2021-04-06  102.88  104.12  100.45  104.12         0.00 %   

     Volume Turn Over  
1157      -         -  
1156      -         -  
1155      -         -  
1154      -         -  
1153      -         -  


In [3]:
df['tp'] = (df['Close']+df['High']+df['Low'])/3

In [4]:
# Clean thousands separators before numeric conversion
df['Volume'] = pd.to_numeric(df['Volume'].astype(str).str.replace(',', '', regex=False), errors='coerce')
df['rmf'] = df['tp'] * df['Volume']

In [5]:
#Find postive and negative money flow
df['tp_diff'] = df['tp'].diff()
df['Postive_tp']= df['tp_diff'] > 0
df['Negative_tp']= df['tp_diff'] < 0


In [6]:
df['pos_flow'] = df['rmf'].where(df['Postive_tp'], 0)
df['neg_flow'] = df['rmf'].where(df['Negative_tp'], 0)

pos_sum = df['pos_flow'].rolling(14).sum()
neg_sum = df['neg_flow'].rolling(14).sum()

mfr = pos_sum / neg_sum
df['mfi'] = 100 - (100 / (1 + mfr))

In [7]:
df.tail()

,Symbol,Date,Open,High,Low,Close,Percent Change,Volume,Turn Over,tp,rmf,tp_diff,Postive_tp,Negative_tp,pos_flow,neg_flow,mfi
4,INVESTMENT,2026-03-24,113.19,114.05,112.42,113.24,0.11 %,6.612420e+08,-,113.236667,7.487684e+10,0.706667,True,False,7.487684e+10,0.000000e+00,57.480898
3,INVESTMENT,2026-03-25,113.17,114.26,112.39,112.63,-0.54 %,1.034589e+09,-,113.093333,1.170052e+11,-0.143333,False,True,0.000000e+00,1.170052e+11,49.342409
2,INVESTMENT,2026-03-26,113.25,113.27,111.60,113.21,0.51 %,1.323563e+09,-,112.693333,1.491567e+11,-0.400000,False,True,0.000000e+00,1.491567e+11,43.096713
1,INVESTMENT,2026-03-29,113.22,113.45,109.79,109.83,-2.99 %,7.595668e+08,-,111.023333,8.432964e+10,-1.670000,False,True,0.000000e+00,8.432964e+10,38.805134
0,INVESTMENT,2026-03-30,109.82,110.44,107.98,108.08,-1.59 %,7.393248e+08,-,108.833333,8.046318e+10,-2.190000,False,True,0.000000e+00,8.046318e+10,36.050911


In [8]:
# Prepare data for plotting: convert date to datetime
df_plot = df.copy()
df_plot['Date'] = pd.to_datetime(df_plot['Date'])

In [9]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create subplots with 2 rows and shared x-axis
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    row_heights=[0.7, 0.3],
    subplot_titles=("Candlestick Chart", "Money Flow Index (MFI)")
)

# Convert date to string to use as categorical axis (preserves gaps)
df_plot['Date_str'] = df_plot['Date'].astype(str)

# Add candlestick chart to the first subplot
fig.add_trace(
    go.Candlestick(
        x=df_plot['Date_str'],
        open=df_plot['Open'],
        high=df_plot['High'],
        low=df_plot['Low'],
        close=df_plot['Close'],
        name='OHLC',
        increasing_line_color='green',
        decreasing_line_color='red',
    ),
    row=1, col=1
)

# Add MFI line chart to the second subplot
fig.add_trace(
    go.Scatter(
        x=df_plot['Date_str'],
        y=df_plot['mfi'],
        mode='lines',
        name='MFI',
        line=dict(color='blue', width=2),
        fill='tozeroy',
        fillcolor='rgba(0, 0, 255, 0.2)'
    ),
    row=2, col=1
)

# Add horizontal line at 70 and 30 for overbought/oversold
fig.add_hline(y=70, line_dash="dash", line_color="red", annotation_text="Overbought", row=2)
fig.add_hline(y=30, line_dash="dash", line_color="green", annotation_text="Oversold", row=2)

# Update layout
fig.update_layout(
    height=700,
    title_text="Trading View with MFI Indicator",
    xaxis_rangeslider_visible=False,
    hovermode='x unified',
    template='plotly_white'
)

# Update y-axes labels
fig.update_yaxes(title_text="Price", row=1, col=1)
fig.update_yaxes(title_text="MFI", row=2, col=1)

fig.show()

In [10]:
# Swing detection function
def find_swings(series, lookback=10):
    highs, lows = [], []
    for i in range(lookback, len(series) - lookback):
        window = series.iloc[i-lookback: lookback + i +1]
        current = float(series.iloc[i])
        if current == float(window.max()):
            highs.append(i)
        if current == float(window.min()):
            lows.append(i)
    return highs, lows

# Find swing highs and lows for price
high_idx, low_idx = find_swings(df_plot["Close"])

# Divergence detection
regular_bullish, regular_bearish = [], []
hidden_bullish, hidden_bearish = [], []

# Regular Bullish: Price LL, MFI HL
for i in range(1, len(low_idx)):
    p1, p2 = low_idx[i - 1], low_idx[i]
    if df_plot["Close"].iloc[p2] < df_plot["Close"].iloc[p1] and df_plot["mfi"].iloc[p2] > df_plot["mfi"].iloc[p1]:
        regular_bullish.append((p1, p2))

# Regular Bearish: Price HH, MFI LH
for i in range(1, len(high_idx)):
    p1, p2 = high_idx[i - 1], high_idx[i]
    if df_plot["Close"].iloc[p2] > df_plot["Close"].iloc[p1] and df_plot["mfi"].iloc[p2] < df_plot["mfi"].iloc[p1]:
        regular_bearish.append((p1, p2))

# Hidden Bullish: Price HL, MFI LL
for i in range(1, len(low_idx)):
    p1, p2 = low_idx[i-1], low_idx[i]
    if df_plot["Close"].iloc[p2] > df_plot["Close"].iloc[p1] and df_plot["mfi"].iloc[p2] < df_plot["mfi"].iloc[p1]:
        hidden_bullish.append((p1, p2))

# Hidden Bearish: Price LH, MFI HH
for i in range(1, len(high_idx)):
    p1, p2 = high_idx[i-1], high_idx[i]
    if df_plot["Close"].iloc[p2] < df_plot["Close"].iloc[p1] and df_plot["mfi"].iloc[p2] > df_plot["mfi"].iloc[p1]:
        hidden_bearish.append((p1, p2))

print("Regular Bullish Divergences:", regular_bullish)
print("Regular Bearish Divergences:", regular_bearish)
print("Hidden Bullish Divergences:", hidden_bullish)
print("Hidden Bearish Divergences:", hidden_bearish)

Regular Bullish Divergences: [(368, 379), (460, 482), (691, 732), (967, 994)]
Regular Bearish Divergences: [(190, 205), (536, 553), (644, 659), (755, 799), (1065, 1084), (1119, 1143)]
Hidden Bullish Divergences: [(417, 460), (617, 691), (732, 766), (766, 825), (841, 855), (875, 934), (1059, 1075)]
Hidden Bearish Divergences: [(232, 302), (387, 399), (447, 485), (697, 755), (1037, 1065)]


In [11]:
# Plot divergence lines on the chart
fig_div = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    row_heights=[0.7, 0.3],
    subplot_titles=("Candlestick Chart with Divergences", "Money Flow Index (MFI)")
)

# Add candlestick chart
fig_div.add_trace(
    go.Candlestick(
        x=df_plot['Date_str'],
        open=df_plot['Open'],
        high=df_plot['High'],
        low=df_plot['Low'],
        close=df_plot['Close'],
        name='OHLC',
        increasing_line_color='green',
        decreasing_line_color='red',
    ),
    row=1, col=1
)

# Add MFI
fig_div.add_trace(
    go.Scatter(
        x=df_plot['Date_str'],
        y=df_plot['mfi'],
        mode='lines',
        name='MFI',
        line=dict(color='blue', width=2)
      
    ),
    row=2, col=1
)

# Regular Bullish on PRICE
for p1, p2 in regular_bullish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['Close'].iloc[p1], df_plot['Close'].iloc[p2]],
            mode='lines+markers',
            line=dict(color='green', dash='dash', width=2),
            showlegend=False,
            hovertemplate='Regular Bullish<extra></extra>'
        ),
        row=1, col=1
    )

# Regular Bearish on PRICE
for p1, p2 in regular_bearish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['Close'].iloc[p1], df_plot['Close'].iloc[p2]],
            mode='lines+markers',
            line=dict(color='red', dash='dash', width=2),
            showlegend=False,
            hovertemplate='Regular Bearish<extra></extra>'
        ),
        row=1, col=1
    )

# Hidden Bullish on PRICE
for p1, p2 in hidden_bullish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['Close'].iloc[p1], df_plot['Close'].iloc[p2]],
            mode='lines+markers',
            line=dict(color='lime', dash='dash', width=2),
            showlegend=False,
            hovertemplate='Hidden Bullish<extra></extra>'
        ),
        row=1, col=1
    )

# Hidden Bearish on PRICE
for p1, p2 in hidden_bearish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['Close'].iloc[p1], df_plot['Close'].iloc[p2]],
            mode='lines+markers',
            line=dict(color='orange', dash='dash', width=2),
            showlegend=False,
            hovertemplate='Hidden Bearish<extra></extra>'
        ),
        row=1, col=1
    )

# Regular Bullish on MFI
for p1, p2 in regular_bullish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['mfi'].iloc[p1], df_plot['mfi'].iloc[p2]],
            mode='lines',
            line=dict(color='green', dash='dash', width=2),
            showlegend=False,
            hovertemplate='Regular Bullish<extra></extra>'
        ),
        row=2, col=1
    )

# Regular Bearish on MFI
for p1, p2 in regular_bearish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['mfi'].iloc[p1], df_plot['mfi'].iloc[p2]],
            mode='lines',
            line=dict(color='red', dash='dash', width=2),
            showlegend=False,
            hovertemplate='Regular Bearish<extra></extra>'
        ),
        row=2, col=1
    )

# Hidden Bullish on MFI
for p1, p2 in hidden_bullish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['mfi'].iloc[p1], df_plot['mfi'].iloc[p2]],
            mode='lines',
            line=dict(color='lime', dash='dot', width=2),
            showlegend=False,
            hovertemplate='Hidden Bullish<extra></extra>'
        ),
        row=2, col=1
    )

# Hidden Bearish on MFI
for p1, p2 in hidden_bearish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['mfi'].iloc[p1], df_plot['mfi'].iloc[p2]],
            mode='lines',
            line=dict(color='orange', dash='dot', width=2),
            showlegend=False,
            hovertemplate='Hidden Bearish<extra></extra>'
        ),
        row=2, col=1
    )

# Add overbought/oversold lines (faded)
fig_div.add_hline(y=70, line_dash="dash", line_color="rgba(128, 128, 128, 0.2)", row=2)
fig_div.add_hline(y=30, line_dash="dash", line_color="rgba(128, 128, 128, 0.2)", row=2)

# Add legend entries manually
fig_div.add_trace(go.Scatter(x=[None], y=[None], mode='lines', 
                         line=dict(color='green', dash='dash', width=2), 
                         name='Regular Bullish'), row=1, col=1)
fig_div.add_trace(go.Scatter(x=[None], y=[None], mode='lines', 
                         line=dict(color='red', dash='dash', width=2), 
                         name='Regular Bearish'), row=1, col=1)
fig_div.add_trace(go.Scatter(x=[None], y=[None], mode='lines', 
                         line=dict(color='lime', dash='dot', width=2), 
                         name='Hidden Bullish'), row=1, col=1)
fig_div.add_trace(go.Scatter(x=[None], y=[None], mode='lines', 
                         line=dict(color='orange', dash='dot', width=2), 
                         name='Hidden Bearish'), row=1, col=1)

# Update layout
fig_div.update_layout(
    height=700,
    width=1400,
    title_text="Trading View with MFI Divergences",
    xaxis_rangeslider_visible=False,
    hovermode='x unified',
    template='plotly_white',
    legend=dict(x=0, y=1)
)

fig_div.update_yaxes(title_text="Price", row=1, col=1)
fig_div.update_yaxes(title_text="MFI", row=2, col=1)

# Print summary
print(f"Regular Bullish Divergences: {len(regular_bullish)}")
print(f"Regular Bearish Divergences: {len(regular_bearish)}")
print(f"Hidden Bullish Divergences: {len(hidden_bullish)}")
print(f"Hidden Bearish Divergences: {len(hidden_bearish)}")

fig_div.show()

Regular Bullish Divergences: 4
Regular Bearish Divergences: 6
Hidden Bullish Divergences: 7
Hidden Bearish Divergences: 5


In [12]:
y_r_bullish = [y for _, y in regular_bullish]
y_r_bearish = [y for _, y in regular_bearish]
y_h_bullish = [y for _, y in hidden_bullish]
y_h_bearish = [y for _, y in hidden_bearish]

In [13]:
candle_intervals= [5,10,20,40,60]
results=[]

In [14]:
all_data=[]
for index in y_r_bullish:
    all_data.append((index, "Regular Bullish"))
for index in y_r_bearish:
    all_data.append((index, "Regular Bearish"))
for index in y_h_bullish:
    all_data.append((index, "Hidden Bullish"))
for index in y_h_bearish:
    all_data.append((index, "Hidden Bearish"))
    

In [15]:
symbol= df["Symbol"][0]
symbol

'INVESTMENT'

In [16]:
for i,pattern in all_data:
    if i+max(candle_intervals)>= len(df):
        continue
    for j in candle_intervals:
        price_now = df["Close"].iloc[i]
        price_future = df["Close"].iloc[i+j]

        pc= (price_future-price_now)/price_now *100

        results.append({
            "Sector": symbol,
            "Pattern":pattern,
            "Interval":j,
            "Start Index":i,
            "EndIndex": i+j,
            "Price Now":price_now,
            "Price After Interval":price_future,
            "Percentage Change": pc
        })
        

In [17]:
results = pd.DataFrame(results)

In [18]:
results

,Sector,Pattern,Interval,Start Index,EndIndex,Price Now,Price After Interval,Percentage Change
0,INVESTMENT,Regular Bullish,5,379,384,55.08,58.33,5.900508
1,INVESTMENT,Regular Bullish,10,379,389,55.08,57.34,4.103123
2,INVESTMENT,Regular Bullish,20,379,399,55.08,57.93,5.174292
3,INVESTMENT,Regular Bullish,40,379,419,55.08,60.96,10.675381
4,INVESTMENT,Regular Bullish,60,379,439,55.08,66.29,20.352215
...,...,...,...,...,...,...,...,...
100,INVESTMENT,Hidden Bearish,5,1065,1070,106.15,103.43,-2.562412
101,INVESTMENT,Hidden Bearish,10,1065,1075,106.15,101.77,-4.126236
102,INVESTMENT,Hidden Bearish,20,1065,1085,106.15,105.92,-0.216675
103,INVESTMENT,Hidden Bearish,40,1065,1105,106.15,102.49,-3.447951


In [19]:
results.to_excel(f"{symbol}_MFI_Divergence.xlsx",index=False)